In [ ]:
# =====================================================
# Celda para Google Colab - App Ají/Rocoto con Cloudflare Tunnel
# (CÓDIGO CORREGIDO - sin errores de sintaxis)
# =====================================================

import os
import subprocess
import sys
import time
import re
import getpass

# ------------------------------------------------------------------
# 1. Configuración de claves y credenciales
# ------------------------------------------------------------------
print("🔑 Configuración de API Key de Groq (obligatoria para usar IA)")
api_key = getpass.getpass("Ingresa tu GROQ_API_KEY (o presiona Enter para omitir IA): ")
if api_key:
    os.environ["GROQ_API_KEY"] = api_key
    print("✅ API Key configurada")
else:
    print("⚠️ No se configuró API Key. Las funciones de IA mostrarán mensaje de advertencia.")

print("\n🌍 Configuración opcional de Google Earth Engine")
use_ee = input("¿Usar Earth Engine real? (s/n): ").lower() == 's'
if use_ee:
    print("Sube tu archivo JSON de cuenta de servicio de GEE:")
    from google.colab import files
    uploaded = files.upload()
    json_file = list(uploaded.keys())[0]
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = json_file
    print("✅ Credenciales GEE configuradas")

# ------------------------------------------------------------------
# 2. Instalación de dependencias
# ------------------------------------------------------------------
print("\n📦 Instalando paquetes Python necesarios...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "streamlit", "geopandas", "pandas", "numpy", "matplotlib",
                "folium", "streamlit-folium", "branca", "shapely",
                "earthengine-api", "groq", "pillow", "scipy"])
print("✅ Dependencias instaladas")

# ------------------------------------------------------------------
# 3. Descarga de cloudflared
# ------------------------------------------------------------------
print("\n🌐 Descargando cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ cloudflared listo")

# ------------------------------------------------------------------
# 4. Escribir el archivo app.py (código completo CORREGIDO)
# ------------------------------------------------------------------
app_code = '''
# app.py - Plataforma de Gestión de Riesgos Climáticos para Ají y Rocoto
# Versión completa - CORREGIDA (sin errores de sintaxis)

import streamlit as st
import geopandas as gpd
import pandas as pd
import numpy as np
import tempfile
import os
import zipfile
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from io import BytesIO
from shapely.geometry import Polygon
import math
import folium
from folium.plugins import Fullscreen, MeasureControl
from streamlit_folium import folium_static
from branca.colormap import LinearColormap
from typing import Dict

# ================= AGROIA GEE =================
try:
    from agroia_gee import (
        obtener_ndvi_actual, obtener_ndwi_actual, obtener_ndre_actual,
        obtener_temperatura_actual, obtener_precipitacion_actual,
        obtener_serie_temporal_ndvi, obtener_serie_temporal_temperatura,
        obtener_serie_temporal_precipitacion
    )
    AGROIA_OK = True
except ImportError:
    AGROIA_OK = False

# ================= GOOGLE EARTH ENGINE =================
try:
    import ee
    GEE_AVAILABLE = True
except ImportError:
    GEE_AVAILABLE = False
    st.warning("⚠️ earthengine-api no instalado")

# ================= GROQ IA =================
try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False
    st.warning("⚠️ groq no instalado")

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
if GROQ_API_KEY and GROQ_AVAILABLE:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ================= INICIALIZACIÓN DE GEE =================
def inicializar_gee():
    if not GEE_AVAILABLE:
        return False
    creds_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if creds_path and os.path.exists(creds_path):
        try:
            with open(creds_path, 'r') as f:
                import json
                creds_info = json.load(f)
            credentials = ee.ServiceAccountCredentials(
                creds_info['client_email'],
                key_data=json.dumps(creds_info)
            )
            ee.Initialize(credentials, project=creds_info.get('project_id', 'ee-mawucano25'))
            st.session_state.gee_authenticated = True
            return True
        except Exception as e:
            st.warning(f"Error con cuenta de servicio: {e}")
    try:
        ee.Initialize(project='ee-mawucano25')
        st.session_state.gee_authenticated = True
        return True
    except Exception as e:
        st.session_state.gee_authenticated = False
        st.warning(f"GEE no autenticado: {e}")
        return False

if 'gee_authenticated' not in st.session_state:
    st.session_state.gee_authenticated = False
    if GEE_AVAILABLE:
        inicializar_gee()

# ================= FUNCIONES DE CARGA =================
def validar_crs(gdf):
    if gdf is None or len(gdf) == 0:
        return gdf
    try:
        if gdf.crs is None:
            gdf = gdf.set_crs('EPSG:4326')
        elif str(gdf.crs).upper() != 'EPSG:4326':
            gdf = gdf.to_crs('EPSG:4326')
        return gdf
    except:
        return gdf

def calcular_superficie(gdf):
    try:
        gdf_proj = gdf.to_crs('EPSG:3857')
        area_m2 = gdf_proj.geometry.area.sum()
        return area_m2 / 10000
    except:
        return 0.0

def cargar_shapefile_desde_zip(zip_file):
    try:
        with tempfile.TemporaryDirectory() as tmp_dir:
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(tmp_dir)
            shp_files = [f for f in os.listdir(tmp_dir) if f.endswith('.shp')]
            if shp_files:
                shp_path = os.path.join(tmp_dir, shp_files[0])
                gdf = gpd.read_file(shp_path)
                gdf = validar_crs(gdf)
                return gdf
            else:
                st.error("❌ No se encontró archivo .shp en el ZIP")
                return None
    except Exception as e:
        st.error(f"❌ Error cargando ZIP: {e}")
        return None

def parsear_kml_manual(contenido_kml):
    try:
        root = ET.fromstring(contenido_kml)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        polygons = []
        for polygon_elem in root.findall('.//kml:Polygon', namespaces):
            coords_elem = polygon_elem.find('.//kml:coordinates', namespaces)
            if coords_elem is not None and coords_elem.text:
                coord_list = []
                for coord_pair in coords_elem.text.strip().split():
                    parts = coord_pair.split(',')
                    if len(parts) >= 2:
                        coord_list.append((float(parts[0]), float(parts[1])))
                if len(coord_list) >= 3:
                    polygons.append(Polygon(coord_list))
        if polygons:
            return gpd.GeoDataFrame({'geometry': polygons}, crs='EPSG:4326')
        return None
    except:
        return None

def cargar_kml(kml_file):
    try:
        if kml_file.name.endswith('.kmz'):
            with tempfile.TemporaryDirectory() as tmp_dir:
                with zipfile.ZipFile(kml_file, 'r') as zip_ref:
                    zip_ref.extractall(tmp_dir)
                kml_files = [f for f in os.listdir(tmp_dir) if f.endswith('.kml')]
                if kml_files:
                    kml_path = os.path.join(tmp_dir, kml_files[0])
                    with open(kml_path, 'r', encoding='utf-8') as f:
                        contenido = f.read()
                    gdf = parsear_kml_manual(contenido)
                    if gdf is not None:
                        return gdf
        else:
            contenido = kml_file.read().decode('utf-8')
            gdf = parsear_kml_manual(contenido)
            if gdf is not None:
                return gdf
        kml_file.seek(0)
        gdf = gpd.read_file(kml_file)
        gdf = validar_crs(gdf)
        return gdf
    except Exception as e:
        st.error(f"❌ Error cargando KML/KMZ: {e}")
        return None

def cargar_archivo_parcela(uploaded_file):
    try:
        if uploaded_file.name.endswith('.zip'):
            gdf = cargar_shapefile_desde_zip(uploaded_file)
        elif uploaded_file.name.endswith(('.kml', '.kmz')):
            gdf = cargar_kml(uploaded_file)
        elif uploaded_file.name.endswith('.geojson'):
            gdf = gpd.read_file(uploaded_file)
            gdf = validar_crs(gdf)
        else:
            st.error("Formato no soportado. Use ZIP, KML, KMZ o GeoJSON.")
            return None
        if gdf is not None:
            gdf = validar_crs(gdf)
            gdf = gdf.explode(ignore_index=True)
            gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
            if len(gdf) == 0:
                st.error("No se encontraron polígonos.")
                return None
            geom_unida = gdf.unary_union
            gdf_unido = gpd.GeoDataFrame({'geometry': [geom_unida]}, crs='EPSG:4326')
            st.info(f"✅ Se unieron {len(gdf)} polígonos.")
            return gdf_unido
        return None
    except Exception as e:
        st.error(f"❌ Error cargando archivo: {e}")
        return None

# ================= DIVISIÓN EN BLOQUES Y NDVI =================
def dividir_parcela_en_bloques(gdf, n_bloques):
    if gdf is None or len(gdf) == 0:
        return gdf
    gdf = validar_crs(gdf)
    parcela = gdf.iloc[0].geometry
    bounds = parcela.bounds
    minx, miny, maxx, maxy = bounds
    n_cols = math.ceil(math.sqrt(n_bloques))
    n_rows = math.ceil(n_bloques / n_cols)
    width = (maxx - minx) / n_cols
    height = (maxy - miny) / n_rows
    bloques = []
    for i in range(n_rows):
        for j in range(n_cols):
            if len(bloques) >= n_bloques:
                break
            cell_minx = minx + j*width
            cell_maxx = minx + (j+1)*width
            cell_miny = miny + i*height
            cell_maxy = miny + (i+1)*height
            cell_poly = Polygon([(cell_minx,cell_miny),(cell_maxx,cell_miny),
                                 (cell_maxx,cell_maxy),(cell_minx,cell_maxy)])
            inter = parcela.intersection(cell_poly)
            if not inter.is_empty and inter.area > 0:
                bloques.append(inter)
    if bloques:
        return gpd.GeoDataFrame({'id_bloque': range(1,len(bloques)+1), 'geometry': bloques}, crs='EPSG:4326')
    return gdf

def obtener_ndvi_por_bloque(gdf_bloques, fecha):
    if not GEE_AVAILABLE or not st.session_state.gee_authenticated:
        return [np.nan]*len(gdf_bloques)
    region = ee.Geometry.Rectangle(gdf_bloques.total_bounds.tolist())
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(region)
           .filterDate(fecha.strftime('%Y-%m-%d'), (fecha+timedelta(days=30)).strftime('%Y-%m-%d'))
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30))
           .sort('CLOUDY_PIXEL_PERCENTAGE'))
    if col.size().getInfo() == 0:
        col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterBounds(region)
               .filterDate((fecha-timedelta(days=60)).strftime('%Y-%m-%d'), fecha.strftime('%Y-%m-%d'))
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',70))
               .sort('CLOUDY_PIXEL_PERCENTAGE'))
    ndvi_img = col.first().normalizedDifference(['B8','B4'])
    def get_mean(geom):
        mean_dict = ndvi_img.reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=10, maxPixels=1e9).getInfo()
        return mean_dict.get('nd', None)
    valores = []
    for _, row in gdf_bloques.iterrows():
        geom_ee = ee.Geometry.Polygon(list(row.geometry.exterior.coords))
        media = get_mean(geom_ee)
        valores.append(media if media is not None else np.nan)
    return valores

def calcular_recomendaciones_npk(ndvi, cultivo):
    umbrales = UMBRALES[cultivo]
    ndvi_min = umbrales['NDVI_min']
    if ndvi >= ndvi_min:
        nivel = "Óptimo"; n=p=k=0
    elif ndvi >= ndvi_min*0.75:
        nivel = "Medio"; n=40; p=20; k=30
    else:
        nivel = "Crítico"; n=80; p=40; k=60
    if cultivo == "ROCOTO":
        n = int(n*1.2); k = int(k*1.3)
    return {'nivel':nivel, 'N':n, 'P':p, 'K':k}

def estimar_potencial_cosecha(ndvi, cultivo, area_ha):
    if cultivo == "AJÍ":
        base=18; opt=0.6
    else:
        base=22; opt=0.65
    factor = max(0.3, min(1.2, ndvi/opt))
    rend = base*factor
    total = rend*area_ha
    return round(rend,1), round(total,1)

def consultar_groq(prompt, max_tokens=600, model="llama-3.3-70b-versatile"):
    api_key = os.getenv("GROQ_API_KEY","")
    if not api_key or not GROQ_AVAILABLE:
        return "⚠️ IA no disponible. Configura GROQ_API_KEY."
    try:
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[{"role":"user","content":prompt}],
            max_tokens=max_tokens,
            temperature=0.5
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Error: {str(e)}"

def generar_recomendaciones_agroecologicas(cultivo, fase, ndvi, temp, humedad, precip):
    principios = "1. Reciclaje de nutrientes\\n2. Salud del suelo\\n3. Diversificación\\n4. Sinergias\\n5. Resiliencia climática\\n6. Conocimiento local\\n7. Gobernanza\\n8. Economía circular\\n9. Bienestar\\n10. Paisajes sostenibles"
    prompt = f"Eres agroecólogo. Para {cultivo} fase {fase}, NDVI={ndvi:.2f}, temp={temp:.1f}°C, humedad={humedad:.2f}, precip={precip:.1f} mm. Genera recomendación concreta para cada principio: {principios}"
    return consultar_groq(prompt, max_tokens=800)

def generar_plan_agroecologico_completo(cultivo, fase, ndvi, temp, humedad, precip, area_ha):
    prompt = f"Plan agroecológico para {cultivo}, {area_ha:.1f} ha, fase {fase}, NDVI={ndvi:.2f}, temp={temp:.1f}°C, humedad={humedad:.2f}, precip={precip:.1f} mm. Incluye manejo de suelo, control biológico, diversificación, agua, insumos ecológicos, monitoreo."
    return consultar_groq(prompt, max_tokens=800)

def generar_alerta_detallada(fase, ndvi, temp, precip, humedad, cultivo, umbrales):
    prompt = f"Alerta agronómica para {cultivo} fase {fase}: NDVI={ndvi:.2f}, temp={temp:.1f}°C, precip={precip:.1f} mm, humedad={humedad:.2f}. Evalúa riesgo, causas y da 3 recomendaciones."
    return consultar_groq(prompt, max_tokens=600)

class CalculadorCarbono:
    def __init__(self):
        self.factores = {'conversion_carbono':0.47, 'ratio_co2':3.67, 'ratio_raiz':0.24,
                         'proporcion_madera_muerta':0.05, 'acumulacion_hojarasca':2.0, 'carbono_suelo':1.5}
    def calcular_carbono_hectarea(self, ndvi, precip_anual):
        factor = min(1.6, max(0.7, precip_anual/1200))
        if ndvi>0.7: agb = (15+(ndvi-0.7)*80)*factor
        elif ndvi>0.5: agb = (8+(ndvi-0.5)*60)*factor
        elif ndvi>0.3: agb = (4+(ndvi-0.3)*40)*factor
        else: agb = (2+ndvi*20)*factor
        agb = min(45,max(3,agb))
        C_agb = agb*0.47
        C_bgb = C_agb*(0.24*0.6)
        C_dw = C_agb*0.05
        C_li = 2.0*0.4*0.47
        C_soc = 1.5*(0.8+factor*0.2)
        total = C_agb+C_bgb+C_dw+C_li+C_soc
        co2 = total*3.67
        return {'carbono_total_ton_ha':round(total,2), 'co2_equivalente_ton_ha':round(co2,2),
                'desglose':{'AGB':round(C_agb,2),'BGB':round(C_bgb,2),'DW':round(C_dw,2),'LI':round(C_li,2),'SOC':round(C_soc,2)}}

def estimar_precipitacion_promedio(df_precip):
    if df_precip is not None and not df_precip.empty and 'precip' in df_precip.columns:
        return df_precip['precip'].mean()*365
    return 1200.0

CULTIVOS = ["AJÍ","ROCOTO","PAPA ANDINA"]
ICONOS = {"AJÍ":"🌶️","ROCOTO":"🥵","PAPA ANDINA":"🥔"}
UMBRALES = {
    "AJÍ":{"NDVI_min":0.4,"NDRE_min":0.15,"temp_min":18,"temp_max":30,"humedad_min":0.25,"humedad_max":0.65},
    "ROCOTO":{"NDVI_min":0.45,"NDRE_min":0.18,"temp_min":16,"temp_max":28,"humedad_min":0.30,"humedad_max":0.70},
    "PAPA ANDINA":{"NDVI_min":0.5,"NDRE_min":0.20,"temp_min":10,"temp_max":22,"humedad_min":0.35,"humedad_max":0.75}
}

st.set_page_config(page_title="Gestión de Riesgos Climáticos - Ají y Rocoto", layout="wide")
st.title("🌶️ Plataforma de Gestión de Riesgos Climáticos")
st.markdown("---")

with st.sidebar:
    st.header("⚙️ Configuración")
    cultivo = st.selectbox("Cultivo", CULTIVOS)
    uploaded_file = st.file_uploader("Subir parcela (GeoJSON, KML, KMZ, ZIP)", type=['geojson','kml','kmz','zip'])
    fecha_fin = st.date_input("Fecha fin", datetime.now())
    fecha_inicio = st.date_input("Fecha inicio", datetime.now() - timedelta(days=90))
    fase_fenologica = st.selectbox("Fase actual", ["siembra","desarrollo","floracion","fructificacion","cosecha"])
    n_bloques = st.slider("Número de bloques (NPK)", 1, 64, 16)
    st.caption(f"GEE: {'✅ Autenticado' if st.session_state.get('gee_authenticated',False) else '❌ No autenticado'}")

if not uploaded_file:
    st.info("👈 Sube un archivo de parcela para comenzar.")
    st.stop()

with st.spinner("Cargando parcela..."):
    gdf = cargar_archivo_parcela(uploaded_file)
    if gdf is None:
        st.stop()
    area_ha = calcular_superficie(gdf)
    st.success(f"✅ Parcela cargada: {area_ha:.2f} ha")

ndvi_val, temp_val, humedad_val, precip_actual = 0.55, 24.0, 0.45, 3.2
df_precip = pd.DataFrame({'date':[datetime.now()],'precip':[3.0]})

tab1, tab2, tab3, tab4, tab5, tab6, tab7, tab8, tab9 = st.tabs(
    ["📊 Dashboard","🗺️ Mapa","📈 Monitoreo","🌾 Fertilidad","🌱 Agroecología",
     "🌍 Carbono","⚠️ Alertas","📄 Gobernanza","💾 Exportar"])

with tab1:
    st.header("Dashboard")
    c1,c2,c3,c4 = st.columns(4)
    c1.metric("NDVI", f"{ndvi_val:.2f}")
    c2.metric("Temperatura", f"{temp_val:.1f}°C")
    c3.metric("Humedad", f"{humedad_val:.2f}")
    c4.metric("Fase", fase_fenologica.capitalize())
    st.line_chart(pd.Series([0.5,0.55,0.6], index=['Ene','Feb','Mar']))

with tab2:
    st.header("Mapa de Riesgo")
    st.info("Mapa interactivo completo disponible en versión local.")

with tab3:
    st.header("Monitoreo")
    st.metric("NDVI", ndvi_val)
    st.metric("Temperatura", f"{temp_val}°C")

with tab4:
    st.header("Fertilidad NPK")
    if st.button("Calcular por bloque"):
        if not st.session_state.gee_authenticated:
            st.warning("GEE no disponible. Usando simulación.")
            gdf_bloques = dividir_parcela_en_bloques(gdf, n_bloques)
            if gdf_bloques is not None:
                ndvis = [0.5 + np.random.randn()*0.1 for _ in range(len(gdf_bloques))]
                gdf_bloques['ndvi'] = ndvis
                areas = [calcular_superficie(gpd.GeoDataFrame({'geometry':[row.geometry]}, crs='EPSG:4326')) for _,row in gdf_bloques.iterrows()]
                gdf_bloques['area'] = areas
                recs = [calcular_recomendaciones_npk(v, cultivo) for v in ndvis]
                gdf_bloques['N'] = [r['N'] for r in recs]
                gdf_bloques['P'] = [r['P'] for r in recs]
                gdf_bloques['K'] = [r['K'] for r in recs]
                gdf_bloques['rend'] = [estimar_potencial_cosecha(v, cultivo, a)[0] for v,a in zip(ndvis, areas)]
                st.dataframe(gdf_bloques[['id_bloque','area','ndvi','N','P','K','rend']].round(2))
                st.download_button("Descargar CSV", gdf_bloques.to_csv(index=False), "fertilidad.csv")
        else:
            gdf_bloques = dividir_parcela_en_bloques(gdf, n_bloques)
            if gdf_bloques is not None:
                ndvis = obtener_ndvi_por_bloque(gdf_bloques, fecha_fin)
                gdf_bloques['ndvi'] = ndvis
                areas = [calcular_superficie(gpd.GeoDataFrame({'geometry':[row.geometry]}, crs='EPSG:4326')) for _,row in gdf_bloques.iterrows()]
                gdf_bloques['area'] = areas
                recs = [calcular_recomendaciones_npk(v, cultivo) for v in ndvis]
                gdf_bloques['N'] = [r['N'] for r in recs]
                gdf_bloques['P'] = [r['P'] for r in recs]
                gdf_bloques['K'] = [r['K'] for r in recs]
                gdf_bloques['rend'] = [estimar_potencial_cosecha(v, cultivo, a)[0] for v,a in zip(ndvis, areas)]
                st.dataframe(gdf_bloques[['id_bloque','area','ndvi','N','P','K','rend']].round(2))
                st.download_button("Descargar CSV", gdf_bloques.to_csv(index=False), "fertilidad.csv")

with tab5:
    st.header("Agroecología")
    if st.button("Recomendaciones por principio"):
        rec = generar_recomendaciones_agroecologicas(cultivo, fase_fenologica, ndvi_val, temp_val, humedad_val, precip_actual)
        st.markdown(rec)
    if st.button("Plan completo"):
        plan = generar_plan_agroecologico_completo(cultivo, fase_fenologica, ndvi_val, temp_val, humedad_val, precip_actual, area_ha)
        st.markdown(plan)

with tab6:
    st.header("Carbono y Créditos")
    calc = CalculadorCarbono()
    precip_anual = estimar_precipitacion_promedio(df_precip)
    res = calc.calcular_carbono_hectarea(ndvi_val, precip_anual)
    c1,c2,c3,c4 = st.columns(4)
    c1.metric("Carbono", f"{res['carbono_total_ton_ha']} t C/ha")
    c2.metric("CO₂e", f"{res['co2_equivalente_ton_ha']} t/ha")
    co2_total = res['co2_equivalente_ton_ha']*area_ha
    creditos = co2_total/1000
    c3.metric("Créditos", f"{creditos:.1f}")
    c4.metric("Valor USD", f"${creditos*15:,.0f}")
    st.dataframe(pd.DataFrame(res['desglose'].items(), columns=['Pool','t C/ha']))
    st.download_button("Exportar carbono", pd.DataFrame([res]).to_csv(index=False), "carbono.csv")

with tab7:
    if st.button("Generar alerta IA"):
        alerta = generar_alerta_detallada(fase_fenologica, ndvi_val, temp_val, precip_actual, humedad_val, cultivo, UMBRALES[cultivo])
        st.markdown(alerta)

with tab8:
    st.markdown("**Gobernanza para ají/rocoto**\\n- Comité de gestión\\n- Monitoreo mensual\\n- Capacitación continua")

with tab9:
    if st.button("Exportar GeoJSON"):
        st.download_button("Descargar", gdf.to_json(), "parcela.geojson")

st.caption("Plataforma integrada con fertilidad NPK, agroecología y cálculo de carbono.")
'''

# Escribir el archivo
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
print("✅ app.py escrito correctamente")

# ------------------------------------------------------------------
# 5. Lanzar Streamlit
# ------------------------------------------------------------------
print("\n🚀 Iniciando Streamlit...")
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(10)

# ------------------------------------------------------------------
# 6. Crear túnel Cloudflare
# ------------------------------------------------------------------
print("🌐 Creando túnel Cloudflare...")
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(6)

url = None
for line in iter(tunnel_proc.stderr.readline, b''):
    line = line.decode()
    if "https://" in line and ".trycloudflare.com" in line:
        match = re.search(r'(https://[a-z0-9\-]+\.trycloudflare\.com)', line)
        if match:
            url = match.group(1)
            print(f"\n🎉 APLICACIÓN DISPONIBLE EN: {url}\n")
            break

if not url:
    print("⚠️ No se pudo obtener la URL. Revisa logs de cloudflared.")

print("Presiona Ctrl+C para detener...")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    streamlit_proc.terminate()
    tunnel_proc.terminate()
    print("App detenida.")

🔑 Configuración de API Key de Groq (obligatoria para usar IA)
✅ API Key configurada

🌍 Configuración opcional de Google Earth Engine
Sube tu archivo JSON de cuenta de servicio de GEE:


Saving democultivos-ce9a27a59d9e.json to democultivos-ce9a27a59d9e.json
✅ Credenciales GEE configuradas

📦 Instalando paquetes Python necesarios...
✅ Dependencias instaladas

🌐 Descargando cloudflared...
✅ cloudflared listo
✅ app.py escrito correctamente

🚀 Iniciando Streamlit...
🌐 Creando túnel Cloudflare...

🎉 APLICACIÓN DISPONIBLE EN: https://sphere-collapse-samuel-monica.trycloudflare.com

Presiona Ctrl+C para detener...
